# M3 — DEA: Análise Envoltória de Dados

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Caso: Qual hospital municipal é mais eficiente?

A Secretaria Estadual de Saúde do PR contratou a Gradus para identificar quais dos 5 hospitais municipais de médio porte estão sendo mais eficientes em transformar recursos em serviços.

**Inputs (3):** orçamento anual (R\$ M), leitos, médicos FTE.  
**Outputs (2):** atendimentos/ano, altas/ano.

Não podemos usar uma simples razão (output/input) porque há múltiplos inputs e múltiplos outputs.

**DEA** resolve isso: para cada DMU $k$, monta um PL que tenta combinar os outros DMUs para produzir o que $k$ produz com **menos inputs**. Se conseguir, $k$ é ineficiente.

## Setup — dois solvers lado a lado

OR-Tools (open-source) + Gurobi (comercial, Genoa representa no Brasil). DEA é puramente LP — ambos resolvem trivialmente, e o ponto interessante é **quando você roda 100s de DMUs em sequência, a velocidade do Gurobi compensa**.

In [ ]:
%pip install -q ortools gurobipy pandas

In [ ]:
from ortools.linear_solver import pywraplp
import pandas as pd
import numpy as np

# Os 5 hospitais municipais
df = pd.DataFrame({
    'hospital': ['Maringá', 'Cascavel', 'Londrina', 'Ponta Grossa', 'Foz do Iguaçu'],
    'orc':     [18, 22, 28, 16, 24],     # R$ M
    'leitos':  [120, 140, 180, 100, 150],
    'medicos': [80, 95, 130, 70, 110],
    'atend':   [95, 100, 145, 85, 105],  # mil/ano
    'altas':   [12, 13, 18, 10, 14],     # mil/ano
})

INPUTS  = ['orc', 'leitos', 'medicos']
OUTPUTS = ['atend', 'altas']
df

## Modelo CCR-input-oriented

Para cada hospital $k$ (DMU em avaliação), resolvemos:

$$\min\ \theta_k \quad \text{s.a.}$$

- Para cada input $i$: $\sum_j \lambda_j\, x_{ij} \le \theta_k\, x_{ik}$
- Para cada output $r$: $\sum_j \lambda_j\, y_{rj} \ge y_{rk}$
- $\lambda_j \ge 0$

Interpretação:
- $\theta_k = 1$: hospital $k$ está na fronteira (eficiente)
- $\theta_k < 1$: ineficiente — poderia produzir o mesmo com $\theta_k$ dos inputs atuais
- $\lambda_j > 0$: hospital $j$ é peer (benchmark) de $k$

In [ ]:
def dea_ccr_input(df, inputs, outputs):
    n = len(df)
    resultados = []
    for k in range(n):
        solver = pywraplp.Solver.CreateSolver('GLOP')
        theta = solver.NumVar(0, 1, 'theta')
        lam   = [solver.NumVar(0, solver.infinity(), f'lam[{j}]') for j in range(n)]

        # Inputs: combinação ≤ θ · input do avaliado
        for inp in inputs:
            x = df[inp].values
            solver.Add(sum(lam[j] * x[j] for j in range(n)) <= theta * x[k])

        # Outputs: combinação ≥ output do avaliado
        for o in outputs:
            y = df[o].values
            solver.Add(sum(lam[j] * y[j] for j in range(n)) >= y[k])

        solver.Minimize(theta)
        solver.Solve()

        peers = {df['hospital'][j]: lam[j].solution_value()
                 for j in range(n) if lam[j].solution_value() > 1e-6}
        resultados.append({
            'hospital': df['hospital'][k],
            'theta': theta.solution_value(),
            'peers': peers,
        })
    return resultados

res = dea_ccr_input(df, INPUTS, OUTPUTS)
for r in res:
    peers_str = ', '.join(f'{p}={l:.3f}' for p, l in r['peers'].items())
    status = '✓ Eficiente' if r['theta'] >= 0.9999 else f'⚠ Ineficiente ({(1-r["theta"])*100:.1f}%)'
    print(f"{r['hospital']:>16}: θ = {r['theta']:.4f}  {status}")
    print(f"{'':>16}   Peers: {peers_str}")

### O mesmo DEA em Gurobi

Mesma estrutura — 1 LP por DMU, em loop. Quando a amostra cresce (centenas de agências bancárias, milhares de lojas), o tempo agregado de todos os LPs vira mensurável; Gurobi resolve cada um em poucos ms.

Vamos também comparar tempos OR-Tools (GLOP) vs Gurobi nos 15 hospitais.

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import time

def dea_ccr_gurobi(df, inputs, outputs):
    n = len(df)
    resultados = []
    for k in range(n):
        m = gp.Model(f'DEA_{k}')
        m.Params.OutputFlag = 0
        theta = m.addVar(lb=0, ub=1, name='theta')
        lam = m.addVars(range(n), lb=0, name='lam')
        for inp in inputs:
            x = df[inp].values
            m.addConstr(gp.quicksum(lam[j] * x[j] for j in range(n)) <= theta * x[k])
        for o in outputs:
            y = df[o].values
            m.addConstr(gp.quicksum(lam[j] * y[j] for j in range(n)) >= y[k])
        m.setObjective(theta, GRB.MINIMIZE)
        m.optimize()
        peers = {df['hospital'][j]: lam[j].X for j in range(n) if lam[j].X > 1e-6}
        resultados.append({'hospital': df['hospital'][k], 'theta': theta.X, 'peers': peers})
    return resultados

# Benchmark: comparar tempos OR-Tools vs Gurobi nos 15 hospitais
t0 = time.time()
res_or = dea_ccr_input(df15, INPUTS, OUTPUTS)
t_or = time.time() - t0

t0 = time.time()
res_gb = dea_ccr_gurobi(df15, INPUTS, OUTPUTS)
t_gb = time.time() - t0

print(f"Tempo OR-Tools (15 LPs):  {t_or*1000:.1f} ms")
print(f"Tempo Gurobi   (15 LPs):  {t_gb*1000:.1f} ms")
print(f"Speed-up Gurobi: {t_or/t_gb:.1f}x")

# Confirmar que chegam ao mesmo resultado
for r_o, r_g in zip(res_or, res_gb):
    assert abs(r_o['theta'] - r_g['theta']) < 1e-4, f"Discrepância em {r_o['hospital']}"
print('\n✓ Os dois solvers chegam exatamente nos mesmos θ.')

### Discussão

- **Maringá, Londrina e Ponta Grossa** estão na fronteira ($\theta = 1$).
- **Cascavel** ($\theta = 0,929$) poderia produzir os mesmos outputs com 92,9 % dos inputs — ~7 % de "folga" relativa. Seu peer é Maringá com $\lambda = 1,083$.
- **Foz do Iguaçu** ($\theta = 0,933$) idem — Maringá também é seu peer.

Maringá é o **benchmark dominante** — os ineficientes deveriam estudar como Maringá organiza recursos.

**Atenção:** DEA é *comparativo*. "Eficiente" aqui não significa "melhor possível em teoria" — só significa que nenhum outro DMU desta amostra produz mais com menos. Se você tirar Maringá, a fronteira muda.

---

## Exercício de extensão: 15 hospitais + BCC + super-eficiência

Vamos ampliar a amostra para 15 hospitais (os 5 originais + 10 sintéticos) e introduzir três técnicas que aparecem no uso prático de DEA:

### 1. Modelo BCC (retornos variáveis de escala)

Diferença vs CCR: adiciona a restrição $\sum_j \lambda_j = 1$. Significa que a combinação linear dos peers deve ter "tamanho" similar ao avaliado.

- **CCR** assume retornos *constantes* de escala — dobrar inputs deveria dobrar outputs.
- **BCC** permite retornos *variáveis* — eficiência pura, separando do efeito escala.

$$\text{Eficiência de escala} = \frac{\theta_{CCR}}{\theta_{BCC}}$$

Se BCC = 1 mas CCR < 1, o hospital é tecnicamente eficiente mas está no tamanho "errado" (escala subótima).

In [ ]:
# Amostra estendida: 15 hospitais
extra = pd.DataFrame({
    'hospital': [f'H{i:02d}' for i in range(6, 16)],
    'orc':     [20, 14, 30, 19, 26, 22, 17, 32, 21, 15],
    'leitos':  [130, 95, 200, 125, 165, 135, 110, 210, 130, 100],
    'medicos': [100, 65, 145, 88, 120, 95, 75, 155, 92, 68],
    'atend':   [110, 80, 150, 100, 120, 105, 88, 160, 112, 75],
    'altas':   [14, 9, 19, 13, 16, 14, 11, 20, 14, 10],
})
df15 = pd.concat([df, extra], ignore_index=True)

def dea_generico(df, inputs, outputs, model='CCR'):
    n = len(df)
    out = []
    for k in range(n):
        solver = pywraplp.Solver.CreateSolver('GLOP')
        theta = solver.NumVar(0, 1, 'theta')
        lam = [solver.NumVar(0, solver.infinity(), f'lam[{j}]') for j in range(n)]
        for inp in inputs:
            x = df[inp].values
            solver.Add(sum(lam[j] * x[j] for j in range(n)) <= theta * x[k])
        for o in outputs:
            y = df[o].values
            solver.Add(sum(lam[j] * y[j] for j in range(n)) >= y[k])
        if model == 'BCC':
            solver.Add(sum(lam[j] for j in range(n)) == 1)
        solver.Minimize(theta)
        solver.Solve()
        out.append({'hospital': df['hospital'][k], 'theta': theta.solution_value()})
    return pd.DataFrame(out)

ccr15 = dea_generico(df15, INPUTS, OUTPUTS, 'CCR').rename(columns={'theta': 'θ_CCR'})
bcc15 = dea_generico(df15, INPUTS, OUTPUTS, 'BCC').rename(columns={'theta': 'θ_BCC'})
comp = ccr15.merge(bcc15, on='hospital')
comp['scale_eff'] = comp['θ_CCR'] / comp['θ_BCC']
comp.round(4)

### Leitura do CCR vs BCC

Observe que **CCR é sempre mais restritivo** que BCC (resolve um problema com menos liberdade), então θ_CCR ≤ θ_BCC para qualquer DMU.

- DMU com **CCR < BCC = 1**: tecnicamente eficiente, mas problema de escala (está pequeno ou grande demais).
- DMU com **CCR = BCC = 1**: eficiência total — operando na escala ótima.
- DMU com **CCR = BCC < 1**: ineficiente puramente por *má utilização* de recursos, não por escala.

### 2. Análise de slack (TODO)

Mesmo com θ = 1, pode sobrar folga em alguma dimensão. Para detectar, rode uma **segunda etapa**: dado θ ótimo, maximize a soma dos slacks. Se > 0, a DMU é "pseudo-eficiente".

In [ ]:
# TODO: implementar a segunda etapa para detectar slacks
# Para cada DMU com θ = 1:
#   max  sum(s_inp) + sum(s_out)
#   s.a.  sum(lam_j · x_ij) + s_inp_i = θ · x_ik   ∀i
#         sum(lam_j · y_rj) - s_out_r = y_rk        ∀r
#         lam ≥ 0, s ≥ 0
# Se sum(slacks) > 0: DMU pseudo-eficiente

### 3. Super-eficiência (Andersen-Petersen)

Para *rankear* os eficientes (todos têm θ = 1, então CCR/BCC empata), rodamos o modelo **excluindo cada DMU do seu próprio conjunto de referência**:

$$\min\ \theta_k^{AP} \quad \text{s.a.}\quad \sum_{j \ne k} \lambda_j x_{ij} \le \theta_k x_{ik},\ \sum_{j \ne k} \lambda_j y_{rj} \ge y_{rk}$$

Agora $\theta_k^{AP}$ pode ser > 1 (interpretação: quanto $k$ é "mais eficiente" que os demais). Ordena a fronteira.

In [ ]:
def super_eficiencia(df, inputs, outputs):
    n = len(df)
    out = []
    for k in range(n):
        solver = pywraplp.Solver.CreateSolver('GLOP')
        theta = solver.NumVar(0, solver.infinity(), 'theta')  # pode ser > 1 agora
        lam = [solver.NumVar(0, solver.infinity(), f'lam[{j}]') if j != k
               else solver.NumVar(0, 0, f'lam[{j}]')  # zera o próprio
               for j in range(n)]
        for inp in inputs:
            x = df[inp].values
            solver.Add(sum(lam[j] * x[j] for j in range(n)) <= theta * x[k])
        for o in outputs:
            y = df[o].values
            solver.Add(sum(lam[j] * y[j] for j in range(n)) >= y[k])
        solver.Minimize(theta)
        status = solver.Solve()
        val = theta.solution_value() if status == pywraplp.Solver.OPTIMAL else None
        out.append({'hospital': df['hospital'][k], 'theta_AP': val})
    return pd.DataFrame(out)

# Aplica super-eficiência só nos eficientes do CCR
eficientes_ccr = ccr15[ccr15['θ_CCR'] >= 0.9999]['hospital'].tolist()
print(f'Eficientes pelo CCR ({len(eficientes_ccr)}): {eficientes_ccr}')

df_efs = df15[df15['hospital'].isin(eficientes_ccr)].reset_index(drop=True)
ap = super_eficiencia(df_efs, INPUTS, OUTPUTS)
print()
print('Ranking por super-eficiência (maior = mais robusto):')
print(ap.sort_values('theta_AP', ascending=False).round(4).to_string(index=False))

### Para discutir

1. **CCR vs BCC nos 5 originais:** quais hospitais têm ineficiência de escala?
2. **Peer analysis na amostra grande:** quais hospitais são peers "dominantes" (aparecem como benchmark para muitos)?
3. **Super-eficiência:** dentre os hospitais eficientes pelo CCR, qual é o mais "robusto" (maior $\theta_{AP}$)?
4. **Quando usar DEA na consultoria:** lista de situações típicas — agências bancárias, lojas de varejo, unidades escolares, ESFs, distribuidoras. O que precisa ser homogêneo?